# Notebook for testing path generation using trajgenpy


_Imports and Helpers_


In [ ]:
from mirte_lc_nav2.navigator_types import SystematicNavigator
from mirte_lc_nav2.navigators import BousPath, SpiralPath
import mirte_lc_nav2.utils as ut
import cv2


def show_img(img):
    img = cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX)
    img = img.astype("uint8")

    plt.figure(figsize=(6, 6))
    plt.imshow(img, cmap="gray")
    plt.title("Normalized Map")
    plt.axis("off")
    plt.show()

## Setup

In order to demonstrate the different navigators, a map must first be loaded into the notebook. \
We use a map provided by the [mirte navigation](https://github.com/MartijnWisse/mirte_navigation) ros2 package


In [ ]:
path = "maps/lab_map.pgm"
map_img = cv2.imread(path, cv2.IMREAD_UNCHANGED)

show_img(map_img)

## Systematic Navigator

Each defined navigator is a derived class of `SystematicNavigator`, a metaclass encompassing all navigators \
that plan coverage paths over the whole free space area. Standard for the derived classes is that they \
update their internal map and as a result plan a path over this map.

this is done in three steps:

- Normalization of the costmap
- Identification of free space boundaries
- Conversion of these boundaries into one polygon


In [ ]:
navigator = SystematicNavigator()

navigator.update_map(map_img)
show_img(navigator.binary_costmap)

contours = navigator.contours
contour_img = cv2.drawContours(map_img, contours, -1, (0, 0, 255), 2)
show_img(contour_img)

## BousPath

This navigator performs boustropethaan cellular decomposition (bcd) on the free space of the map.
It then generates swaths and inter-swath paths called headlands over each cell.
Then finally it plans a path between each cell to return a complete coverage path over the entire free space.
